# AoC 2024 Day 4 — Ceres Search

**Spark lesson: modelling a grid as a cell relation**

Puzzle: <https://adventofcode.com/2024/day/4>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

---

## The puzzle

A grid of letters — a word search.

- **Part 1** — count every occurrence of `XMAS`, in all 8 directions: horizontal, vertical, both diagonals, forwards and backwards. Occurrences may overlap.
- **Part 2** — despite the name, find `X-MAS`: two `MAS` (each forwards or backwards) crossing diagonally on a shared central `A`.

## The Spark angle

The move that makes this tractable in Spark is **refusing to treat the grid as a grid.**

Explode it into a `(row, col, char)` relation and "look in direction (dr,dc)" becomes an **equi-join on offset coordinates**: join cell *(r,c)* to cell *(r+dr, c+dc)*. Chain one join per letter and a surviving row is a complete match.

Searching all 8 directions at once is then a **cross join against an 8-row directions table** — Spark broadcasts something that small, so the search stays three joins deep no matter how many directions you add. That is the payoff of the relational framing: the direction count moved from *code structure* into *data*.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day04

spark = get_spark('aoc-2024-day04')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = """\
MMMSXXMASM
MSAMXMSMSA
AMXSXMAAMM
MSAMASMSMX
XMASAMXAMM
XXAMMXXAMA
SMSMSASXSS
SAXAMASAAA
MAMMMXMMMM
MXMXAXMASX
"""

print('part 1:', day04.part1(spark, EXAMPLE), '(expected 18)')
print('part 2:', day04.part2(spark, EXAMPLE), '(expected 9)')

### From grid to relation

The whole trick is the first cell below. Once the grid is a table of coordinates, everything after it is ordinary SQL.

In [ ]:
from pyspark.sql import functions as F

grid = day04.cells(spark, EXAMPLE)
grid.show(8)
print('cells:', grid.count())

# One direction, one step: every X with an M immediately to its right.
xs = grid.filter(F.col('ch') == 'X').select('r', 'c')
ms = grid.filter(F.col('ch') == 'M').select(F.col('r').alias('nr'), F.col('c').alias('nc'))
xs.join(ms, (F.col('nr') == F.col('r')) & (F.col('nc') == F.col('c') + 1)).show()

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 4)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

for part in (1, 2):
    fn = getattr(day04, f'part{part}')
    started = time.perf_counter()
    answer = fn(spark, data)
    print(f'part {part}: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Notes & gotchas

- `split(line, '')` emits a **trailing empty string**, which is why `cells()` filters `ch != ''`. Leave it in and every row gains a phantom cell.
- The grid is `cache()`d because both parts join against it repeatedly — without it Spark re-explodes the source on every join.
- Part 2's name is a trap: it is *not* about the letters `XMAS` at all.
- Worth trying: rewrite part 1 to count each direction in a separate query and sum. Same answer, ~8× the query plans — a concrete demonstration of why the directions table is worth it.